El **CAPM de Sharpe-Lintner** establece que el retorno esperado de un activo es función lineal de su exposición al riesgo sistemático (beta). En este módulo estimamos los betas y alfas de cada acción usando la cartera de mercado cap-weighted y la cartera tangente calculada en el módulo anterior.

In [1]:
#| label: setup-capm
#| code-fold: true
#| code-summary: "Datos y configuración"

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11})

TICKERS = ['AAPL','MSFT','AMZN','GOOGL','META','JPM','BAC','GS','WFC','MS',
           'JNJ','PFE','UNH','MRK','ABBV','XOM','CVX','KO','PG','WMT']
RF_MONTHLY = 0.0448 / 12

raw = yf.download(TICKERS, start='2015-01-01', end='2024-12-31',
                  interval='1mo', auto_adjust=True, progress=False)['Close']
returns = raw.dropna(axis=1, thresh=int(0.9*len(raw))).pct_change().dropna()

mu = returns.mean().values
V  = returns.cov().values
N  = len(mu)
iota = np.ones(N)
V_inv = np.linalg.inv(V)

# Pesos del portafolio cap-weighted (aproximación equiponderada como proxy)
w_mkt = np.ones(N) / N   # equiponderado como proxy de mercado
r_mkt = (returns * w_mkt).sum(axis=1)

# Cartera tangente
excess = mu - RF_MONTHLY
z_q = V_inv @ excess
w_q = z_q / (iota @ z_q)
r_q = (returns.values @ w_q)

print(f"Activos: {N} | Observaciones: {len(returns)}")

Activos: 20 | Observaciones: 119


### Betas y alfas

El **beta** de un activo mide su sensibilidad al movimiento del mercado:

$$\beta_j = \frac{\text{Cov}(r_j, r_m)}{\text{Var}(r_m)}$$

El **alfa** es el retorno en exceso no explicado por el mercado. Bajo el CAPM de Sharpe-Lintner, $\alpha_j = 0$ para todos los activos:

$$\alpha_j = \bar{z}_j - \beta_j \bar{z}_m \qquad \text{donde } z = r - R_f$$

In [2]:
#| label: betas-alfas

var_mkt = r_mkt.var()
tickers_avail = returns.columns.tolist()

results = []
for i, t in enumerate(tickers_avail):
    r_j = returns.iloc[:, i]
    beta_j  = r_j.cov(r_mkt) / var_mkt
    z_j     = (r_j - RF_MONTHLY).mean()
    z_m     = (r_mkt - RF_MONTHLY).mean()
    alpha_j = z_j - beta_j * z_m

    # OLS para p-value del alfa
    z_excess_j = r_j.values - RF_MONTHLY
    z_excess_m = r_mkt.values - RF_MONTHLY
    slope, intercept, r_val, p_val, se = stats.linregress(z_excess_m, z_excess_j)

    results.append({
        'Ticker': t,
        'Beta':   round(beta_j, 4),
        'Alpha (mensual)': round(alpha_j, 5),
        'Alpha p-value':   round(p_val, 4),
        'R²':   round(r_val**2, 4),
        'Retorno esperado CAPM (%)': round((RF_MONTHLY + beta_j * (r_mkt.mean() - RF_MONTHLY)) * 100, 3)
    })

capm_df = pd.DataFrame(results).set_index('Ticker').sort_values('Beta', ascending=False)

print(f"Acción con mayor beta:  {capm_df['Beta'].idxmax()} ({capm_df['Beta'].max():.4f})")
print(f"Acción con menor beta:  {capm_df['Beta'].idxmin()} ({capm_df['Beta'].min():.4f})")
print(f"Acción con mayor alfa:  {capm_df['Alpha (mensual)'].idxmax()} ({capm_df['Alpha (mensual)'].max():.5f})")
print(f"Acción con menor alfa:  {capm_df['Alpha (mensual)'].idxmin()} ({capm_df['Alpha (mensual)'].min():.5f})")
print()
capm_df

Acción con mayor beta:  BAC (1.6516)
Acción con menor beta:  PG (0.4287)
Acción con mayor alfa:  AMZN (0.00978)
Acción con menor alfa:  WFC (-0.00962)



,Beta,Alpha (mensual),Alpha p-value,R²,Retorno esperado CAPM (%)
Ticker,,,,,
BAC,1.6516,-0.00649,0.0,0.6802,2.088
MS,1.5505,-0.00295,0.0,0.6278,1.983
GS,1.5332,-0.00457,0.0,0.6400,1.965
WFC,1.3787,-0.00962,0.0,0.5084,1.805
JPM,1.2930,0.00019,0.0,0.6227,1.716
CVX,1.2545,-0.00726,0.0,0.4686,1.676
AMZN,1.1214,0.00978,0.0,0.2960,1.538
XOM,1.1210,-0.00692,0.0,0.3748,1.537
AAPL,1.1027,0.00711,0.0,0.3535,1.518


### Security Market Line (SML)

La SML muestra la relación lineal entre beta y retorno esperado según el CAPM. Activos por encima de la línea tienen alfa positivo (subvalorados); los de abajo, alfa negativo (sobrevalorados).

In [3]:
#| label: sml-plot
#| code-fold: true
#| fig-cap: "Security Market Line (SML) — retorno esperado vs beta"

betas   = capm_df['Beta'].values
mu_real = (returns.mean() * 100).values
mu_capm = capm_df['Retorno esperado CAPM (%)'].values

beta_range = np.linspace(betas.min() - 0.1, betas.max() + 0.1, 100)
sml = (RF_MONTHLY + beta_range * (r_mkt.mean() - RF_MONTHLY)) * 100

fig, ax = plt.subplots(figsize=(11, 7))
ax.plot(beta_range, sml, color='#2563eb', linewidth=2, label='SML (CAPM)')

colors = ['#16a34a' if a > 0 else '#dc2626'
          for a in capm_df['Alpha (mensual)'].values]
ax.scatter(betas, mu_real, c=colors, s=80, zorder=5)

for i, t in enumerate(capm_df.index):
    ax.annotate(t, (betas[i], mu_real[i]),
                fontsize=8, xytext=(4, 3), textcoords='offset points')

from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor='#16a34a', label='α > 0 (retorno por encima del CAPM)'),
    Patch(facecolor='#dc2626', label='α < 0 (retorno por debajo del CAPM)'),
]
ax.legend(handles=[plt.Line2D([0],[0],color='#2563eb',lw=2,label='SML')] + legend_els,
          fontsize=9)

ax.set_xlabel('Beta (β)', fontsize=12)
ax.set_ylabel('Retorno mensual promedio (%)', fontsize=12)
ax.set_title('Security Market Line — CAPM (2015–2024)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

n_above = (capm_df['Alpha (mensual)'] > 0).sum()
print(f"Activos con alfa positivo: {n_above}/{N}")
print(f"Activos con alfa negativo: {N - n_above}/{N}")

Activos con alfa positivo: 10/20
Activos con alfa negativo: 10/20
